# LABORATORIO DE ARITMÉTICA MODULAR - VERSIÓN MEJORADA
## Integrantes: Carlos Eduardo Guzmán Torres

---

Este laboratorio incluye:
1. Operaciones básicas modulares
2. Algoritmo de Euclides extendido
3. **NUEVO:** Conversión modular universal
4. **NUEVO:** Grupo de unidades Z_n*

In [1]:
# IMPORTACIONES
from fractions import Fraction
from math import gcd as builtin_gcd

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 1. OPERACIONES BÁSICAS MODULARES

In [2]:
def modular_addition(a: int, b: int, p: int) -> int:
    """Calcula (a + b) mod p"""
    return (a + b) % p

def modular_subtraction(a: int, b: int, p: int) -> int:
    """Calcula (a - b) mod p"""
    return (a - b) % p

def modular_multiplication(a: int, b: int, p: int) -> int:
    """Calcula (a × b) mod p"""
    return (a * b) % p

# Pruebas
print("=== OPERACIONES BÁSICAS ===")
print(f"Suma: (5 + 3) mod 7 = {modular_addition(5, 3, 7)}")
print(f"Resta: (5 - 3) mod 7 = {modular_subtraction(5, 3, 7)}")
print(f"Multiplicación: (5 × 3) mod 7 = {modular_multiplication(5, 3, 7)}")

=== OPERACIONES BÁSICAS ===
Suma: (5 + 3) mod 7 = 1
Resta: (5 - 3) mod 7 = 2
Multiplicación: (5 × 3) mod 7 = 1


## 2. ALGORITMO DE EUCLIDES EXTENDIDO E INVERSO MODULAR

In [3]:
def extended_gcd(a: int, b: int) -> tuple:
    """
    Algoritmo extendido de Euclides.
    Retorna (gcd, x, y) donde gcd = ax + by
    """
    if a == 0:
        return b, 0, 1
    
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    
    return gcd, x, y

def modular_inverse(a: int, p: int) -> int:
    """
    Calcula el inverso modular de a módulo p.
    Lanza ValueError si no existe.
    """
    gcd, x, y = extended_gcd(a % p, p)
    
    if gcd != 1:
        raise ValueError(f"El inverso de {a} mod {p} no existe (no son coprimos)")
    
    return x % p

# Pruebas
print("=== INVERSO MODULAR ===")
try:
    inv = modular_inverse(3, 7)
    print(f"3^(-1) mod 7 = {inv}")
    print(f"Verificación: (3 × {inv}) mod 7 = {(3 * inv) % 7}")
except ValueError as e:
    print(f"Error: {e}")

=== INVERSO MODULAR ===
3^(-1) mod 7 = 5
Verificación: (3 × 5) mod 7 = 1


## 3. CONVERSIÓN MODULAR UNIVERSAL (NUEVA FUNCIONALIDAD)

Esta función detecta automáticamente el tipo de expresión y calcula su valor módulo m:
- **Enteros positivos**: `64 mod 24`
- **Enteros negativos**: `-3 mod 2`
- **Inversos**: `3^-1 mod 7`
- **Fracciones**: `14/2 mod 9`

In [4]:
def _calcular_entero(expresion: str, m: int) -> dict:
    """Calcula módulo de un entero"""
    numero = int(expresion)
    resultado = numero % m
    
    tipo = "entero_negativo" if numero < 0 else "entero_positivo"
    
    explicacion = {
        'original': numero,
        'division': numero // m,
        'residuo': resultado,
        'proceso': f"{numero} = {m} × {numero // m} + {resultado}"
    }
    
    return {
        'tipo': tipo,
        'resultado': resultado,
        'explicacion': explicacion
    }

def _calcular_inverso(expresion: str, m: int) -> dict:
    """Calcula el inverso multiplicativo"""
    base_str = expresion.replace("^-1", "").replace("^(-1)", "").strip()
    a = int(base_str) % m
    
    mcd = builtin_gcd(a, m)
    
    if mcd != 1:
        return {
            'tipo': 'inverso_no_existe',
            'resultado': None,
            'explicacion': {
                'a': a,
                'mcd': mcd,
                'razon': f"MCD({a}, {m}) = {mcd} ≠ 1"
            }
        }
    
    inverso = modular_inverse(a, m)
    verificacion = (a * inverso) % m
    
    return {
        'tipo': 'inverso_multiplicativo',
        'resultado': inverso,
        'explicacion': {
            'a': a,
            'inverso': inverso,
            'verificacion': f"{a} × {inverso} = {a * inverso} ≡ {verificacion} (mod {m})"
        }
    }

def _calcular_fraccion(expresion: str, m: int) -> dict:
    """Calcula módulo de una fracción"""
    partes = expresion.split("/")
    numerador = int(partes[0].strip())
    denominador = int(partes[1].strip())
    
    fraccion = Fraction(numerador, denominador)
    num = fraccion.numerator
    den = fraccion.denominator
    
    try:
        den_inv = modular_inverse(den, m)
        resultado = (num * den_inv) % m
        
        return {
            'tipo': 'fraccion',
            'resultado': resultado,
            'explicacion': {
                'original': f"{numerador}/{denominador}",
                'simplificada': f"{num}/{den}",
                'den_inverso': den_inv,
                'calculo': f"({num} × {den_inv}) mod {m} = {resultado}"
            }
        }
    except ValueError:
        return {
            'tipo': 'fraccion_no_existe',
            'resultado': None,
            'explicacion': {
                'razon': f"El denominador {den} no tiene inverso mod {m}"
            }
        }

def conversion_modular_universal(expresion: str, modulo: int) -> dict:
    """
    Convierte cualquier expresión a su forma módulo m.
    Detecta automáticamente el tipo de expresión.
    """
    expresion = expresion.strip()
    
    if "^-1" in expresion or "^(-1)" in expresion:
        return _calcular_inverso(expresion, modulo)
    elif "/" in expresion:
        return _calcular_fraccion(expresion, modulo)
    else:
        return _calcular_entero(expresion, modulo)

print("✓ Funciones de conversión cargadas")

✓ Funciones de conversión cargadas


In [5]:
def mostrar_conversion(expresion: str, modulo: int):
    """Muestra la conversión con formato bonito"""
    print(f"\n{'='*60}")
    print(f"CONVERSIÓN: {expresion} mod {modulo}")
    print(f"{'='*60}")
    
    resultado = conversion_modular_universal(expresion, modulo)
    tipo = resultado['tipo'].replace('_', ' ').title()
    
    print(f"\nTipo: {tipo}")
    print(f"\nExplicación:")
    
    exp = resultado['explicacion']
    
    if resultado['tipo'] in ['entero_positivo', 'entero_negativo']:
        print(f"  • Número original: {exp['original']}")
        print(f"  • Proceso: {exp['proceso']}")
        print(f"  • Residuo: {exp['residuo']}")
    
    elif resultado['tipo'] == 'inverso_multiplicativo':
        print(f"  • Número: {exp['a']}")
        print(f"  • Inverso: {exp['inverso']}")
        print(f"  • Verificación: {exp['verificacion']}")
    
    elif resultado['tipo'] == 'inverso_no_existe':
        print(f"  • {exp['razon']}")
        print(f"  • No existe inverso")
    
    elif resultado['tipo'] == 'fraccion':
        print(f"  • Fracción original: {exp['original']}")
        print(f"  • Simplificada: {exp['simplificada']}")
        print(f"  • Inverso del denominador: {exp['den_inverso']}")
        print(f"  • Cálculo: {exp['calculo']}")
    
    elif resultado['tipo'] == 'fraccion_no_existe':
        print(f"  • {exp['razon']}")
    
    if resultado['resultado'] is not None:
        print(f"\n✓ RESULTADO: {expresion} ≡ {resultado['resultado']} (mod {modulo})")
    else:
        print(f"\n✗ NO EXISTE RESULTADO")
    
    print(f"{'='*60}\n")

print("✓ Función de visualización cargada")

✓ Función de visualización cargada


### Pruebas de Conversión Modular Universal

Probando los 4 ejemplos solicitados:

In [6]:
# Test 1: Entero negativo
# Ejemplo: -3 mod 2 = 1
mostrar_conversion("-3", 2)


CONVERSIÓN: -3 mod 2

Tipo: Entero Negativo

Explicación:
  • Número original: -3
  • Proceso: -3 = 2 × -2 + 1
  • Residuo: 1

✓ RESULTADO: -3 ≡ 1 (mod 2)



In [7]:
# Test 2: Entero positivo
# Ejemplo: 64 mod 24 = 16
mostrar_conversion("64", 24)


CONVERSIÓN: 64 mod 24

Tipo: Entero Positivo

Explicación:
  • Número original: 64
  • Proceso: 64 = 24 × 2 + 16
  • Residuo: 16

✓ RESULTADO: 64 ≡ 16 (mod 24)



In [8]:
# Test 3: Inverso multiplicativo
# Ejemplo: 3^-1 mod 7 = 5
mostrar_conversion("3^-1", 7)


CONVERSIÓN: 3^-1 mod 7

Tipo: Inverso Multiplicativo

Explicación:
  • Número: 3
  • Inverso: 5
  • Verificación: 3 × 5 = 15 ≡ 1 (mod 7)

✓ RESULTADO: 3^-1 ≡ 5 (mod 7)



In [9]:
# Test 4: Fracción
# Ejemplo: 14/2 mod 9 = 7
mostrar_conversion("14/2", 9)


CONVERSIÓN: 14/2 mod 9

Tipo: Fraccion

Explicación:
  • Fracción original: 14/2
  • Simplificada: 7/1
  • Inverso del denominador: 1
  • Cálculo: (7 × 1) mod 9 = 7

✓ RESULTADO: 14/2 ≡ 7 (mod 9)



## 4. GRUPO DE UNIDADES Z_n* (NUEVA FUNCIONALIDAD)

Encuentra todos los elementos invertibles de Z_n y calcula sus inversos multiplicativos.

**Ejemplo:** Para Z_12, el grupo de unidades es Z_12* = {1, 5, 7, 11}

In [10]:
def grupo_unidades(n: int) -> dict:
    """
    Calcula el grupo de unidades Z_n*.
    Z_n* = {a ∈ Z_n : MCD(a, n) = 1}
    """
    if n <= 0:
        raise ValueError("n debe ser positivo")
    
    grupo = []
    inversos = {}
    
    for a in range(1, n):
        if builtin_gcd(a, n) == 1:
            grupo.append(a)
            inv = modular_inverse(a, n)
            inversos[a] = inv
    
    tabla = {}
    for a in grupo:
        tabla[a] = {}
        for b in grupo:
            tabla[a][b] = (a * b) % n
    
    return {
        'n': n,
        'grupo': grupo,
        'inversos': inversos,
        'tamaño': len(grupo),
        'tabla': tabla
    }

def mostrar_grupo_unidades(n: int):
    """Muestra el grupo de unidades con formato"""
    print(f"\n{'='*70}")
    print(f"GRUPO DE UNIDADES Z_{n}*")
    print(f"{'='*70}")
    
    resultado = grupo_unidades(n)
    
    print(f"\nConjunto completo: Z_{n} = {{{', '.join(map(str, range(n)))}}}")
    print(f"\nGrupo de unidades (coprimos con {n}):")
    print(f"Z_{n}* = {{{', '.join(map(str, resultado['grupo']))}}}")
    print(f"\nTamaño: |Z_{n}*| = {resultado['tamaño']}")
    
    print(f"\n{'─'*70}")
    print("TABLA DE INVERSOS MULTIPLICATIVOS")
    print(f"{'─'*70}")
    print(f"{'Elemento':>12} │ {'Inverso':>12} │ {'Verificación'}")
    print(f"{'─'*70}")
    
    for elemento in sorted(resultado['grupo']):
        inverso = resultado['inversos'][elemento]
        verif = f"{elemento} × {inverso} ≡ {(elemento * inverso) % n} (mod {n})"
        print(f"{elemento:>12} │ {inverso:>12} │ {verif}")
    
    print(f"{'─'*70}")
    
    if len(resultado['grupo']) <= 12:
        print(f"\nTABLA DE MULTIPLICACIÓN MOD {n}")
        print(f"{'─'*70}")
        
        print("   × ", end="")
        for b in resultado['grupo']:
            print(f"{b:>4}", end="")
        print("\n" + "─"*70)
        
        for a in resultado['grupo']:
            print(f"{a:>4} ", end="")
            for b in resultado['grupo']:
                print(f"{resultado['tabla'][a][b]:>4}", end="")
            print()
    
    print(f"{'='*70}\n")

print("✓ Funciones de grupo de unidades cargadas")

✓ Funciones de grupo de unidades cargadas


### Pruebas de Grupo de Unidades

Probando diferentes módulos:

In [11]:
# Test 1: Z_12*
# Resultado esperado: {1, 5, 7, 11}
mostrar_grupo_unidades(12)


GRUPO DE UNIDADES Z_12*

Conjunto completo: Z_12 = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}

Grupo de unidades (coprimos con 12):
Z_12* = {1, 5, 7, 11}

Tamaño: |Z_12*| = 4

──────────────────────────────────────────────────────────────────────
TABLA DE INVERSOS MULTIPLICATIVOS
──────────────────────────────────────────────────────────────────────
    Elemento │      Inverso │ Verificación
──────────────────────────────────────────────────────────────────────
           1 │            1 │ 1 × 1 ≡ 1 (mod 12)
           5 │            5 │ 5 × 5 ≡ 1 (mod 12)
           7 │            7 │ 7 × 7 ≡ 1 (mod 12)
          11 │           11 │ 11 × 11 ≡ 1 (mod 12)
──────────────────────────────────────────────────────────────────────

TABLA DE MULTIPLICACIÓN MOD 12
──────────────────────────────────────────────────────────────────────
   ×    1   5   7  11
──────────────────────────────────────────────────────────────────────
   1    1   5   7  11
   5    5   1  11   7
   7    7  11   1   5
  11  

In [12]:
# Test 2: Z_10*
# Resultado esperado: {1, 3, 7, 9}
mostrar_grupo_unidades(10)


GRUPO DE UNIDADES Z_10*

Conjunto completo: Z_10 = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}

Grupo de unidades (coprimos con 10):
Z_10* = {1, 3, 7, 9}

Tamaño: |Z_10*| = 4

──────────────────────────────────────────────────────────────────────
TABLA DE INVERSOS MULTIPLICATIVOS
──────────────────────────────────────────────────────────────────────
    Elemento │      Inverso │ Verificación
──────────────────────────────────────────────────────────────────────
           1 │            1 │ 1 × 1 ≡ 1 (mod 10)
           3 │            7 │ 3 × 7 ≡ 1 (mod 10)
           7 │            3 │ 7 × 3 ≡ 1 (mod 10)
           9 │            9 │ 9 × 9 ≡ 1 (mod 10)
──────────────────────────────────────────────────────────────────────

TABLA DE MULTIPLICACIÓN MOD 10
──────────────────────────────────────────────────────────────────────
   ×    1   3   7   9
──────────────────────────────────────────────────────────────────────
   1    1   3   7   9
   3    3   9   1   7
   7    7   1   9   3
   9    9   7   3

In [13]:
# Test 3: Z_16*
# Resultado esperado: {1, 3, 5, 7, 9, 11, 13, 15}
mostrar_grupo_unidades(16)


GRUPO DE UNIDADES Z_16*

Conjunto completo: Z_16 = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15}

Grupo de unidades (coprimos con 16):
Z_16* = {1, 3, 5, 7, 9, 11, 13, 15}

Tamaño: |Z_16*| = 8

──────────────────────────────────────────────────────────────────────
TABLA DE INVERSOS MULTIPLICATIVOS
──────────────────────────────────────────────────────────────────────
    Elemento │      Inverso │ Verificación
──────────────────────────────────────────────────────────────────────
           1 │            1 │ 1 × 1 ≡ 1 (mod 16)
           3 │           11 │ 3 × 11 ≡ 1 (mod 16)
           5 │           13 │ 5 × 13 ≡ 1 (mod 16)
           7 │            7 │ 7 × 7 ≡ 1 (mod 16)
           9 │            9 │ 9 × 9 ≡ 1 (mod 16)
          11 │            3 │ 11 × 3 ≡ 1 (mod 16)
          13 │            5 │ 13 × 5 ≡ 1 (mod 16)
          15 │           15 │ 15 × 15 ≡ 1 (mod 16)
──────────────────────────────────────────────────────────────────────

TABLA DE MULTIPLICACIÓN MOD 16
──────

## 5. CASOS DE PRUEBA ADICIONALES

In [14]:
print("=== CASOS DE PRUEBA ADICIONALES ===")
print("\n--- Test: Todos los grupos de unidades comunes ---")

for n in [5, 7, 8, 9]:
    resultado = grupo_unidades(n)
    print(f"Z_{n}* = {resultado['grupo']}")

=== CASOS DE PRUEBA ADICIONALES ===

--- Test: Todos los grupos de unidades comunes ---
Z_5* = [1, 2, 3, 4]
Z_7* = [1, 2, 3, 4, 5, 6]
Z_8* = [1, 3, 5, 7]
Z_9* = [1, 2, 4, 5, 7, 8]


In [15]:
print("\n--- Test: Casos donde no existe resultado ---")

# Inverso que no existe
mostrar_conversion("2^-1", 4)

# Fracción con denominador sin inverso
mostrar_conversion("4/2", 6)


--- Test: Casos donde no existe resultado ---

CONVERSIÓN: 2^-1 mod 4

Tipo: Inverso No Existe

Explicación:
  • MCD(2, 4) = 2 ≠ 1
  • No existe inverso

✗ NO EXISTE RESULTADO


CONVERSIÓN: 4/2 mod 6

Tipo: Fraccion

Explicación:
  • Fracción original: 4/2
  • Simplificada: 2/1
  • Inverso del denominador: 1
  • Cálculo: (2 × 1) mod 6 = 2

✓ RESULTADO: 4/2 ≡ 2 (mod 6)



## 6. FUNCIÓN INTERACTIVA PARA JUPYTER

In [16]:
def calculadora_interactiva():
    """
    Función interactiva para uso en Jupyter Notebook
    """
    print("\n" + "="*70)
    print("CALCULADORA DE ARITMÉTICA MODULAR")
    print("="*70)
    print("\nFunciones disponibles:")
    print("\n1. CONVERSIÓN MODULAR UNIVERSAL")
    print("   mostrar_conversion(expresion, modulo)")
    print("   Ejemplos:")
    print("     mostrar_conversion('-3', 2)")
    print("     mostrar_conversion('64', 24)")
    print("     mostrar_conversion('3^-1', 7)")
    print("     mostrar_conversion('14/2', 9)")
    print("\n2. GRUPO DE UNIDADES")
    print("   mostrar_grupo_unidades(n)")
    print("   Ejemplos:")
    print("     mostrar_grupo_unidades(12)")
    print("     mostrar_grupo_unidades(10)")
    print("\n3. OPERACIONES BÁSICAS")
    print("   modular_addition(a, b, m)")
    print("   modular_subtraction(a, b, m)")
    print("   modular_multiplication(a, b, m)")
    print("   modular_inverse(a, m)")
    print("="*70)

# Mostrar guía
calculadora_interactiva()


CALCULADORA DE ARITMÉTICA MODULAR

Funciones disponibles:

1. CONVERSIÓN MODULAR UNIVERSAL
   mostrar_conversion(expresion, modulo)
   Ejemplos:
     mostrar_conversion('-3', 2)
     mostrar_conversion('64', 24)
     mostrar_conversion('3^-1', 7)
     mostrar_conversion('14/2', 9)

2. GRUPO DE UNIDADES
   mostrar_grupo_unidades(n)
   Ejemplos:
     mostrar_grupo_unidades(12)
     mostrar_grupo_unidades(10)

3. OPERACIONES BÁSICAS
   modular_addition(a, b, m)
   modular_subtraction(a, b, m)
   modular_multiplication(a, b, m)
   modular_inverse(a, m)


## RESUMEN

### ✅ Funcionalidades Implementadas:

1. **Conversión Modular Universal** - Detecta automáticamente:
   - Enteros positivos: `64 mod 24 = 16`
   - Enteros negativos: `-3 mod 2 = 1`
   - Inversos: `3^-1 mod 7 = 5`
   - Fracciones: `14/2 mod 9 = 7`

2. **Grupo de Unidades Z_n***:
   - Encuentra elementos coprimos con n
   - Calcula todos los inversos multiplicativos
   - Genera tabla de multiplicación
   - Ejemplo: Z_12* = {1, 5, 7, 11}

3. **Operaciones Básicas**:
   - Suma, resta, multiplicación modular
   - Inverso modular
   - Algoritmo de Euclides extendido

---

**Autor:** Carlos Eduardo Guzmán Torres  
**Fecha:** 2025  
**Versión:** 2.0 (Mejorada)